# Fast Pre-filter + BGE + Cross-Encoder Reranker
## No LLM needed — inverted index + structured field matching

**The problem with the previous pre-filter approach:**
- Used an LLM to extract filters → 400ms per query, sometimes wrong
- Stored `summary_keywords` as string in CSV → 2839ms to parse per query
- Result: slower AND worse quality than no filter

**This notebook's approach — fast, deterministic, no LLM:**

```
Query: "software companies in Germany"
              ↓
Step 1: Parse query into tokens  (~0ms)
        → ["software", "companies", "germany"]
              ↓
Step 2: Structured field filter  (~1ms)
        → country == "Germany" (if country mentioned)
        → nace_code starts with "K" (if detectable)
              ↓
Step 3: Inverted index on summary_keywords  (~2ms)
        → pre-built dict: "software" → [idx_1, idx_5, idx_8 ...]
        → intersection with country filter
              ↓
Step 4: BGE searches filtered corpus  (~20ms)
              ↓
Step 5: Cross-encoder reranks top-50  (~263ms)
              ↓
Total: ~290ms  AND  better precision
```

**Folder structure:**
```
result/
└── fast_prefilter/
    ├── inverted_index.json           # Pre-built keyword → company index
    ├── retrieval_filtered.csv        # BGE results on filtered corpus
    ├── reranked_filtered.csv         # After cross-encoder reranking
    ├── evaluation_filtered.csv       # Metrics: Precision, Recall, NDCG @k
    ├── latency_filtered.csv          # Per-query latency breakdown
    ├── comparison_all.csv            # vs all previous methods
    └── plots_filtered.png            # Visualisations
```

## 1 · Environment Setup

In [ ]:
import os, json, time, ast, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import faiss
import torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

load_dotenv()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[Setup] Device        : {DEVICE}')
if torch.cuda.is_available():
    print(f'[Setup] GPU           : {torch.cuda.get_device_name(0)}')
    print(f'[Setup] VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

RESULT_DIR = Path('result/6_fast_prefilter')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/')

# ── Key settings ──────────────────────────────────────────────────────────────
TOP_K_RETRIEVE = 1000   # BGE retrieves this many from filtered corpus
TOP_K_RERANK   = 50     # Cross-encoder reranks this many
MIN_FILTER_SIZE = 100   # Fall back to full corpus if filter leaves fewer than this
print(f'[Setup] BGE retrieves top-{TOP_K_RETRIEVE} | Reranker reranks top-{TOP_K_RERANK}')
print(f'[Setup] Fallback to full corpus if filter leaves < {MIN_FILTER_SIZE} companies')

## 2 · Load Corpus & Fix summary_keywords

The `summary_keywords` column is stored as a **string** in the CSV
(e.g. `"['software', 'tech']"` instead of `['software', 'tech']`).

We fix this once using `ast.literal_eval` so it becomes a real Python list.
This is the bug that caused 2839ms filter time in the previous experiment.

In [ ]:
print('[Load] Loading corpus...')
all_companies = pd.read_csv('result/company_corpus.csv')
print(f'[Load] Corpus size           : {len(all_companies):,} companies')
print(f'[Load] Columns               : {list(all_companies.columns)}')

# ── Fix summary_keywords: string → real list ──────────────────────────────────
print('[Load] Fixing summary_keywords column (string → list)...')
t0 = time.time()

def parse_keywords(val):
    """Safely parse summary_keywords from string to list."""
    if isinstance(val, list):
        return val
    if isinstance(val, str) and val.startswith('['):
        try:
            return ast.literal_eval(val)
        except:
            return []
    return []

all_companies['summary_keywords'] = all_companies['summary_keywords'].apply(parse_keywords)

parse_time = (time.time() - t0) * 1000
sample     = all_companies['summary_keywords'].iloc[0]
print(f'[Load] Parsing took          : {parse_time:.0f}ms')
print(f'[Load] Sample keywords (idx 0): {sample}')
print(f'[Load] Type check            : {type(sample)}')

# ── Check field coverage ─────────────────────────────────────────────────────
print('\n[Load] Field coverage:')
for col in ['country', 'state', 'municipality', 'nace_code',
            'organization_size', 'organization_type', 'summary_keywords']:
    if col in all_companies.columns:
        filled = all_companies[col].apply(
            lambda x: bool(x) if not isinstance(x, float) else False
        ).mean() * 100
        print(f'  {col:<25}: {filled:.1f}% filled')

# ── Load other data ───────────────────────────────────────────────────────────
print('\n[Load] Loading queries and production results...')
production_df = pd.read_excel('dataset/production_results.xlsx')
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries               : {len(data)}')
print(f'[Load] Production rows       : {len(production_df):,}')

## 3 · Build Inverted Index on summary_keywords

An inverted index maps each keyword to the list of company indices that have it.

```python
inverted_index = {
    "software":     [0, 5, 8, 12, ...],   # indices of companies with "software" keyword
    "healthcare":   [3, 7, 11, ...],
    "germany":      [2, 9, 15, ...],
    ...
}
```

**At query time:** look up each query word in the index → intersect the lists → filtered corpus.
This is O(1) per keyword lookup — essentially instant.

In [ ]:
print('[Index] Building inverted index on summary_keywords...')
t0 = time.time()

# keyword (lowercased) → set of company row indices
keyword_index  = defaultdict(set)
# country (lowercased) → set of company row indices
country_index  = defaultdict(set)
# nace letter (e.g. 'K') → set of company row indices
nace_index     = defaultdict(set)
# state (lowercased) → set of company row indices
state_index    = defaultdict(set)
# municipality (lowercased) → set of company row indices
city_index     = defaultdict(set)

for idx, row in all_companies.iterrows():
    # ── summary_keywords index ────────────────────────────────────────────────
    for kw in row['summary_keywords']:
        # index each word in the keyword phrase, not just the full phrase
        for word in kw.lower().split():
            if len(word) > 2:  # skip very short words
                keyword_index[word].add(idx)
        # also index the full phrase
        keyword_index[kw.lower()].add(idx)

    # ── country index ────────────────────────────────────────────────────────
    if pd.notna(row.get('country', None)) and row['country']:
        country_index[str(row['country']).lower()].add(idx)

    # ── nace index — extract just the letter (e.g. 'K' from 'NACE K: ...') ──
    nace_val = row.get('nace_code', None)
    if pd.notna(nace_val) and nace_val:
        match = re.search(r'NACE\s+([A-Z])', str(nace_val))
        if match:
            nace_index[match.group(1)].add(idx)

    # ── state index ──────────────────────────────────────────────────────────
    state_val = row.get('state', None)
    if pd.notna(state_val) and state_val:
        state_index[str(state_val).lower()].add(idx)

    # ── city/municipality index ───────────────────────────────────────────────
    city_val = row.get('municipality', None)
    if pd.notna(city_val) and city_val:
        city_index[str(city_val).lower()].add(idx)

build_time = (time.time() - t0) * 1000

print(f'[Index] Built in             : {build_time:.0f}ms')
print(f'[Index] Unique keywords      : {len(keyword_index):,}')
print(f'[Index] Unique countries     : {len(country_index):,}')
print(f'[Index] Unique NACE codes    : {sorted(nace_index.keys())}')
print(f'[Index] Unique states        : {len(state_index):,}')
print(f'[Index] Unique cities        : {len(city_index):,}')

# ── Quick lookup test ─────────────────────────────────────────────────────────
print(f'\n[Index] Test lookups:')
for word in ['software', 'healthcare', 'germany', 'berlin', 'manufacturing']:
    kw_count  = len(keyword_index.get(word, set()))
    cnt_count = len(country_index.get(word, set()))
    city_count = len(city_index.get(word, set()))
    print(f'  "{word}": keywords={kw_count:,}  country={cnt_count:,}  city={city_count:,}')

## 4 · Fast Filter Function

Parses the query into tokens and applies the inverted index.

**Logic:**
1. Tokenise query (lowercase, remove stopwords)
2. Check each token against country, city, state indexes
3. Check each token and bigrams against keyword index
4. Intersect all matching sets
5. If result too small → fall back to full corpus

**No LLM. No ML. Pure dictionary lookup.**

In [ ]:
# Common English stopwords to ignore when matching keywords
STOPWORDS = {
    'a','an','the','and','or','in','on','at','to','for','of','with',
    'that','this','is','are','was','were','be','been','have','has',
    'do','does','did','will','would','could','should','may','might',
    'companies','company','firms','firm','businesses','business',
    'providers','provider','services','service','solutions','solution',
    'based','focused','providing','offering','specializing','focused',
    'large','small','medium','enterprise','startups','startup',
}

# Known country name variants → canonical
COUNTRY_VARIANTS = {
    'us': 'united states', 'usa': 'united states', 'u.s': 'united states',
    'uk': 'united kingdom', 'u.k': 'united kingdom',
    'de': 'germany', 'deu': 'germany',
    'fr': 'france', 'fra': 'france',
}

# Known NACE code mappings for common industry terms
NACE_HINTS = {
    'software': 'K', 'tech': 'K', 'technology': 'K', 'it': 'K',
    'programming': 'K', 'saas': 'K', 'cloud': 'K', 'data': 'K',
    'ai': 'K', 'ml': 'K', 'fintech': 'K',
    'healthcare': 'Q', 'medical': 'Q', 'pharmaceutical': 'Q', 'health': 'Q',
    'manufacturing': 'C', 'industrial': 'C', 'production': 'C',
    'logistics': 'H', 'transport': 'H', 'shipping': 'H', 'freight': 'H',
    'retail': 'G', 'ecommerce': 'G', 'e-commerce': 'G', 'trade': 'G',
    'construction': 'F', 'building': 'F', 'real estate': 'L',
    'consulting': 'M', 'legal': 'M', 'accounting': 'M',
    'finance': 'K', 'banking': 'K', 'insurance': 'K',
    'agriculture': 'A', 'farming': 'A', 'food': 'C',
    'energy': 'D', 'renewable': 'D', 'solar': 'D', 'wind': 'D',
    'education': 'P', 'training': 'P',
    'media': 'J', 'publishing': 'J', 'entertainment': 'R',
}

def fast_filter(query, verbose=False):
    """
    Apply fast inverted-index pre-filter to the corpus.

    Returns:
        filtered_indices : list of corpus row indices matching the filter
        filter_info      : dict explaining what was matched
    """
    q_lower  = query.lower().strip()
    tokens   = [t for t in re.split(r'[\s,]+', q_lower)
                if t and t not in STOPWORDS and len(t) > 2]

    filter_info   = {'tokens': tokens, 'country': None, 'nace': None,
                     'keywords': [], 'city': None, 'state': None}
    matched_sets  = []

    # ── 1. Country matching ───────────────────────────────────────────────────
    for token in tokens:
        canonical = COUNTRY_VARIANTS.get(token, token)
        if canonical in country_index:
            matched_sets.append(country_index[canonical])
            filter_info['country'] = canonical
            if verbose:
                print(f'  [Filter] Country match: "{canonical}" → {len(country_index[canonical]):,} companies')
            break

    # ── 2. City/municipality matching ─────────────────────────────────────────
    for token in tokens:
        if token in city_index and len(city_index[token]) > 0:
            matched_sets.append(city_index[token])
            filter_info['city'] = token
            if verbose:
                print(f'  [Filter] City match: "{token}" → {len(city_index[token]):,} companies')
            break

    # ── 3. State matching ─────────────────────────────────────────────────────
    for token in tokens:
        if token in state_index and len(state_index[token]) > 0:
            matched_sets.append(state_index[token])
            filter_info['state'] = token
            if verbose:
                print(f'  [Filter] State match: "{token}" → {len(state_index[token]):,} companies')
            break

    # ── 4. NACE code from industry hints ──────────────────────────────────────
    for token in tokens:
        if token in NACE_HINTS:
            nace_letter = NACE_HINTS[token]
            if nace_letter in nace_index:
                matched_sets.append(nace_index[nace_letter])
                filter_info['nace'] = nace_letter
                if verbose:
                    print(f'  [Filter] NACE match: "{token}" → NACE {nace_letter} → {len(nace_index[nace_letter]):,} companies')
                break

    # ── 5. Keyword matching from summary_keywords index ───────────────────────
    kw_matches = []
    for token in tokens:
        if token in keyword_index:
            kw_matches.append(keyword_index[token])
            filter_info['keywords'].append(token)
            if verbose:
                print(f'  [Filter] Keyword match: "{token}" → {len(keyword_index[token]):,} companies')

    # ── 6. Also try bigrams ───────────────────────────────────────────────────
    for i in range(len(tokens) - 1):
        bigram = f'{tokens[i]} {tokens[i+1]}'
        if bigram in keyword_index:
            kw_matches.append(keyword_index[bigram])
            filter_info['keywords'].append(bigram)
            if verbose:
                print(f'  [Filter] Bigram match: "{bigram}" → {len(keyword_index[bigram]):,} companies')

    # Union of all keyword matches (company needs ANY matching keyword)
    if kw_matches:
        kw_union = set().union(*kw_matches)
        matched_sets.append(kw_union)

    # ── 7. Combine: intersection across filter types ──────────────────────────
    # e.g. country=Germany AND keywords contain software
    if not matched_sets:
        # No filters matched at all → return full corpus
        if verbose:
            print(f'  [Filter] No filters matched → full corpus')
        return list(range(len(all_companies))), filter_info

    result = matched_sets[0]
    for s in matched_sets[1:]:
        result = result & s  # intersection

    # ── 8. Fallback if too few companies ──────────────────────────────────────
    if len(result) < MIN_FILTER_SIZE:
        if verbose:
            print(f'  [Filter] Too few companies ({len(result)}) → falling back to full corpus')
        return list(range(len(all_companies))), filter_info

    return sorted(result), filter_info

# ── Test on several queries ───────────────────────────────────────────────────
test_queries = [
    'software companies in Germany',
    'healthcare providers',
    'AI startups in Berlin',
    'manufacturing in Baden-Württemberg',
    'renewable energy companies',
    'logistics companies in the US',
    'B2B SaaS platforms',
]

print('[Filter] === FILTER TEST RESULTS ===')
print(f'{"Query":<45} {"Before":>8} {"After":>8} {"Reduction":>10}  Filters')
print('-' * 100)
for q in test_queries:
    t0       = time.perf_counter()
    indices, info = fast_filter(q)
    ms       = (time.perf_counter() - t0) * 1000
    reduction = 100 * (1 - len(indices) / len(all_companies))
    filters_used = []
    if info['country']:  filters_used.append(f'country={info["country"]}')
    if info['city']:     filters_used.append(f'city={info["city"]}')
    if info['nace']:     filters_used.append(f'nace={info["nace"]}')
    if info['keywords']: filters_used.append(f'kw={info["keywords"][:2]}')
    print(f'{q:<45} {len(all_companies):>8,} {len(indices):>8,} {reduction:>9.1f}%  '
          f'{" | ".join(filters_used)} ({ms:.1f}ms)')

## 5 · Load BGE + Cross-Encoder Models

In [ ]:
print('[Models] Loading BGE embedding model...')
t0 = time.time()
model_bge = SentenceTransformer('BAAI/bge-large-en-v1.5', device=DEVICE)
print(f'[Models] BGE loaded in {time.time()-t0:.1f}s on {DEVICE}')

print('[Models] Loading pre-computed BGE embeddings...')
all_embeddings = np.load('result/3_baseline_BGE/company_embeddings.npy').astype('float32')
print(f'[Models] Embeddings shape    : {all_embeddings.shape}')

print('[Models] Loading cross-encoder reranker...')
t0 = time.time()
tokenizer_reranker = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
model_reranker     = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
model_reranker     = model_reranker.to(DEVICE)
model_reranker.eval()
print(f'[Models] Reranker loaded in {time.time()-t0:.1f}s on {DEVICE}')
print('[Models] All models ready ✅')

## 6 · Full Pipeline Functions

Three functions chained together:
1. `fast_filter` — inverted index filter (already built in §4)
2. `bge_search` — encode query + search filtered FAISS subset
3. `rerank` — cross-encoder scores top-50 candidates

In [ ]:
def bge_search(query, filtered_indices, top_k=TOP_K_RETRIEVE):
    """
    Encode query with BGE and search within the filtered corpus subset.
    Builds a small FAISS index on the fly from pre-computed embeddings.
    """
    # Slice pre-computed embeddings for filtered companies only
    subset_embs = all_embeddings[filtered_indices].astype('float32')

    # Build small FAISS index
    dim   = subset_embs.shape[1]  # 1024
    index = faiss.IndexFlatIP(dim)
    index.add(subset_embs)

    # Encode query
    q_emb = model_bge.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')

    # Search
    k_actual  = min(top_k, len(filtered_indices))
    scores, local_idxs = index.search(q_emb, k_actual)

    # Map local indices back to global corpus indices
    global_idxs = [filtered_indices[i] for i in local_idxs[0]]

    return global_idxs, scores[0].tolist()


def rerank(query, global_idxs, bge_scores, top_k=TOP_K_RERANK, batch_size=32):
    """
    Re-rank top-K BGE candidates using cross-encoder.
    Returns reranked list of (global_idx, reranker_score, bge_score).
    """
    candidates = global_idxs[:top_k]
    summaries  = all_companies.iloc[candidates]['summary'].fillna('').tolist()
    pairs      = [[query, s] for s in summaries]
    all_scores = []

    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch   = pairs[i:i+batch_size]
            encoded = tokenizer_reranker(
                batch, padding=True, truncation=True,
                max_length=512, return_tensors='pt'
            ).to(DEVICE)
            scores = model_reranker(**encoded).logits.squeeze(-1)
            all_scores.extend(scores.cpu().float().tolist())

    # Sort by reranker score
    ranked = sorted(
        zip(candidates, all_scores, bge_scores[:top_k]),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked  # list of (global_idx, reranker_score, bge_score)


# ── Smoke test ────────────────────────────────────────────────────────────────
print('[Test] Running smoke test: "software companies in Germany"')
t0 = time.perf_counter()

test_q   = 'software companies in Germany'
f_idx, f_info = fast_filter(test_q, verbose=True)
g_idx, g_scores = bge_search(test_q, f_idx)
ranked   = rerank(test_q, g_idx, g_scores)

total_ms = (time.perf_counter() - t0) * 1000
print(f'\n[Test] Total pipeline time   : {total_ms:.0f}ms')
print(f'[Test] Corpus after filter   : {len(f_idx):,} / {len(all_companies):,}')
print(f'[Test] BGE retrieved         : {len(g_idx)}')
print(f'[Test] After reranking top-10:')
print(f'  {"Rank":<6} {"Name":<45} {"Reranker":>10} {"BGE":>8}')
print('  ' + '-' * 75)
for rank, (idx, rscore, bscore) in enumerate(ranked[:10], 1):
    name = str(all_companies.iloc[idx]['name'])[:43]
    print(f'  {rank:<6} {name:<45} {rscore:>10.4f} {bscore:>8.4f}')
print('[Test] Smoke test passed ✅')

## 7 · Run Full Pipeline for All 101 Queries

In [ ]:
print(f'[Run] Starting full pipeline for {len(data)} queries...')
print(f'[Run] Filter → BGE (top-{TOP_K_RETRIEVE}) → Reranker (top-{TOP_K_RERANK})')
print('-' * 60)

all_result_rows  = []
all_latency_rows = []
total_start      = time.time()

for i, item in enumerate(data):
    qid   = item['query_id']
    query = item['query']

    # ── Step 1: Fast filter ───────────────────────────────────────────────────
    t0 = time.perf_counter()
    filtered_indices, filter_info = fast_filter(query)
    filter_ms = (time.perf_counter() - t0) * 1000

    # ── Step 2: BGE search on filtered corpus ─────────────────────────────────
    t0 = time.perf_counter()
    global_idxs, bge_scores = bge_search(query, filtered_indices)
    bge_ms = (time.perf_counter() - t0) * 1000

    # ── Step 3: Cross-encoder rerank ──────────────────────────────────────────
    t0 = time.perf_counter()
    reranked = rerank(query, global_idxs, bge_scores)
    rerank_ms = (time.perf_counter() - t0) * 1000

    total_ms = filter_ms + bge_ms + rerank_ms

    # ── Collect results ───────────────────────────────────────────────────────
    for rank, (idx, rscore, bscore) in enumerate(reranked, 1):
        company = all_companies.iloc[idx]
        all_result_rows.append({
            'query_id':       qid,
            'query':          query,
            'rank':           rank,
            'reranker_score': float(rscore),
            'bge_score':      float(bscore),
            'domain':         company['domain'],
            'name':           company['name'],
            'summary':        company['summary'],
        })

    all_latency_rows.append({
        'query_id':          qid,
        'query':             query,
        'corpus_before':     len(all_companies),
        'corpus_after':      len(filtered_indices),
        'reduction_pct':     round(100*(1 - len(filtered_indices)/len(all_companies)), 1),
        'fallback_used':     len(filtered_indices) == len(all_companies),
        'filter_ms':         round(filter_ms, 1),
        'bge_ms':            round(bge_ms, 1),
        'rerank_ms':         round(rerank_ms, 1),
        'total_ms':          round(total_ms, 1),
        'filter_country':    filter_info.get('country', ''),
        'filter_city':       filter_info.get('city', ''),
        'filter_nace':       filter_info.get('nace', ''),
        'filter_keywords':   str(filter_info.get('keywords', [])),
    })

    # ── Progress log every 10 queries ─────────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        avg_ms    = elapsed / (i+1) * 1000
        remaining = (len(data) - i - 1) * elapsed / (i+1)
        print(f'[Run] {i+1:3d}/{len(data)}  |  '
              f'avg {avg_ms:.0f}ms/query  |  '
              f'~{remaining:.0f}s remaining  |  '
              f'last filter: {len(filtered_indices):,} companies')

total_elapsed = time.time() - total_start
print('-' * 60)
print(f'[Run] Done! Total: {total_elapsed:.1f}s  |  '
      f'Avg: {total_elapsed/len(data)*1000:.0f}ms/query')

results_df  = pd.DataFrame(all_result_rows)
latency_df  = pd.DataFrame(all_latency_rows)

results_df.to_csv(RESULT_DIR / 'reranked_filtered.csv',  index=False)
latency_df.to_csv(RESULT_DIR / 'latency_filtered.csv',   index=False)
print(f'[Run] Results saved  : result/fast_prefilter/reranked_filtered.csv')
print(f'[Run] Latency saved  : result/fast_prefilter/latency_filtered.csv')

# ── Filter stats ──────────────────────────────────────────────────────────────
print(f'\n[Run] Filter statistics:')
print(f'  Avg corpus after filter : {latency_df["corpus_after"].mean():,.0f} companies '
      f'({latency_df["reduction_pct"].mean():.1f}% reduction)')
print(f'  Fallback to full corpus : {latency_df["fallback_used"].sum()}/101 queries')
print(f'  Queries using country   : {(latency_df["filter_country"] != "").sum()}/101')
print(f'  Queries using city      : {(latency_df["filter_city"] != "").sum()}/101')
print(f'  Queries using NACE      : {(latency_df["filter_nace"] != "").sum()}/101')
print(f'  Queries using keywords  : {(latency_df["filter_keywords"] != "[]").sum()}/101')

## 8 · Evaluation — Precision, Recall, NDCG@k

In [ ]:
print('[Eval] Starting evaluation...')
K_VALUES = [10, 50, 100, 1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i+2) for i, d in enumerate(retrieved[:k]) if d in relevant)

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

eval_rows = []
for i, item in enumerate(data):
    qid      = item['query_id']
    relevant = get_relevant(qid)

    # Re-ranked top-50 from filtered pipeline
    reranked_domains = (
        results_df[results_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )

    # BGE full results for positions 51-1000 (from baseline)
    bge_full = pd.read_csv('result/3_baseline_BGE/embedding_results.csv')
    bge_remaining_domains = [
        d for d in bge_full[bge_full['query_id']==qid]
        .sort_values('rank')['domain'].tolist()
        if d not in set(reranked_domains)
    ]
    combined = reranked_domains + bge_remaining_domains

    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     item['query'],
            'k':         k,
            'precision': precision_at_k(combined, relevant, k),
            'recall':    recall_at_k(combined, relevant, k),
            'ndcg':      ndcg_at_k(combined, relevant, k),
        })

    if (i+1) % 20 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_filtered.csv', index=False)
print(f'[Eval] Done! Saved to result/fast_prefilter/evaluation_filtered.csv')

## 9 · Full Comparison — All Methods

In [ ]:
print('[Compare] Building final comparison table...')

baseline_df  = pd.read_csv('result/evaluation_fixed.csv')
reranker_df  = pd.read_csv('result/05_reranker/evaluation_reranker.csv')
reranker_df['method'] = 'BGE + Reranker'
eval_df['method']     = 'Filter + BGE + Reranker'

all_methods = {
    'BM25':                    baseline_df[baseline_df['method']=='BM25'],
    'BGE':                     baseline_df[baseline_df['method']=='BGE'],
    'OpenAI':                  baseline_df[baseline_df['method']=='OpenAI'],
    'BGE + Reranker':          reranker_df,
    'Filter + BGE + Reranker': eval_df,
}

print('\n' + '=' * 72)
print(f'{"Method":<28} {"k":>6} | {"NDCG":>7} | {"Prec":>7} | {"Recall":>7}')
print('=' * 72)

comp_rows = []
for method, df in all_methods.items():
    for k in K_VALUES:
        sub  = df[df['k'] == k]
        ndcg = sub['ndcg'].mean()
        prec = sub['precision'].mean()
        rec  = sub['recall'].mean()
        print(f'{method:<28} {k:>6} | {ndcg:>7.3f} | {prec:>7.3f} | {rec:>7.3f}')
        comp_rows.append({'method':method,'k':k,
                          'ndcg':round(ndcg,3),
                          'precision':round(prec,3),
                          'recall':round(rec,3)})
    print('-' * 72)

pd.DataFrame(comp_rows).to_csv(RESULT_DIR/'comparison_all.csv', index=False)
print(f'\n[Compare] Saved to result/fast_prefilter/comparison_all.csv')

# ── Improvements ─────────────────────────────────────────────────────────────
print('\n[Compare] Improvement: Filter+BGE+Reranker vs BGE+Reranker (no filter):')
for k in K_VALUES:
    r_ndcg = reranker_df[reranker_df['k']==k]['ndcg'].mean()
    f_ndcg = eval_df[eval_df['k']==k]['ndcg'].mean()
    diff   = f_ndcg - r_ndcg
    print(f'  NDCG@{k:<5}: Reranker={r_ndcg:.3f}  Filter+Reranker={f_ndcg:.3f}  {diff:+.3f}')

## 10 · Latency Summary

In [ ]:
print('[Latency] Pipeline breakdown:')
avg_filter = latency_df['filter_ms'].mean()
avg_bge    = latency_df['bge_ms'].mean()
avg_rerank = latency_df['rerank_ms'].mean()
avg_total  = latency_df['total_ms'].mean()

BGE_RERANKER_MS = 307.2  # from previous notebook
BGE_ALONE_MS    = 44.4

print(f'\n  Component         | Filter+BGE+Reranker | BGE+Reranker | BGE alone')
print(f'  ──────────────────|─────────────────────|──────────────|──────────')
print(f'  Fast filter       | {avg_filter:>10.1f}ms       |         N/A  |      N/A')
print(f'  BGE encode+search | {avg_bge:>10.1f}ms       |        44.4ms|     44.4ms')
print(f'  Cross-encoder     | {avg_rerank:>10.1f}ms       |       262.8ms|      N/A')
print(f'  ──────────────────|─────────────────────|──────────────|──────────')
print(f'  TOTAL             | {avg_total:>10.1f}ms       |  {BGE_RERANKER_MS:.1f}ms  |     {BGE_ALONE_MS:.1f}ms')

diff = avg_total - BGE_RERANKER_MS
print(f'\n[Latency] Filter+BGE+Reranker vs BGE+Reranker: {diff:+.1f}ms')
if diff < 0:
    print(f'[Latency] Pre-filter SAVED {abs(diff):.1f}ms by reducing BGE search space ✅')
else:
    print(f'[Latency] Pre-filter added {diff:.1f}ms overhead')

print(f'\n[Latency] Corpus reduction per query:')
print(f'  Avg after filter  : {latency_df["corpus_after"].mean():,.0f} / {len(all_companies):,} '
      f'({latency_df["reduction_pct"].mean():.1f}% reduction)')
print(f'  Fallback rate     : {latency_df["fallback_used"].sum()}/101 queries')
print(f'  Min corpus size   : {latency_df["corpus_after"].min():,}')
print(f'  Max corpus size   : {latency_df["corpus_after"].max():,}')

## 11 · Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print('[Plot] Generating plots...')

PLOT_METHODS = ['BM25','BGE','BGE + Reranker','Filter + BGE + Reranker']
colors  = {'BM25':'#2196F3','BGE':'#4CAF50',
            'BGE + Reranker':'#F44336',
            'Filter + BGE + Reranker':'#FF9800'}
markers = {'BM25':'o','BGE':'^',
            'BGE + Reranker':'*',
            'Filter + BGE + Reranker':'D'}
lws     = {'BM25':2,'BGE':2,'BGE + Reranker':2.5,'Filter + BGE + Reranker':3}
mss     = {'BM25':6,'BGE':6,'BGE + Reranker':9,'Filter + BGE + Reranker':8}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for metric, ax, title in [
    ('ndcg',      axes[0], 'NDCG@k'),
    ('precision', axes[1], 'Precision@k'),
    ('recall',    axes[2], 'Recall@k'),
]:
    for m in PLOT_METHODS:
        df   = all_methods[m]
        vals = [df[df['k']==k][metric].mean() for k in K_VALUES]
        ax.plot(K_VALUES, vals, marker=markers[m], color=colors[m],
                label=m, linewidth=lws[m], markersize=mss[m],
                zorder=5 if 'Filter' in m else 3)
    ax.set_xscale('log'); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
    ax.set_xlabel('k'); ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# ── Corpus size distribution ──────────────────────────────────────────────────
ax = axes[3]
no_filter = latency_df[latency_df['fallback_used'] == False]
ax.hist(no_filter['corpus_after'], bins=20, color='#FF9800', edgecolor='white', alpha=0.8)
ax.axvline(len(all_companies), color='#F44336', linestyle='--',
           linewidth=2, label=f'No filter ({len(all_companies):,})')
ax.set_xlabel('Corpus size after filter')
ax.set_ylabel('Number of queries')
ax.set_title('Corpus Reduction per Query\n(fallback queries excluded)', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Filter + BGE + Cross-Encoder Reranker vs Baselines',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULT_DIR/'plots_filtered.png', bbox_inches='tight', dpi=150)
plt.show()
print('[Plot] Saved to result/fast_prefilter/plots_filtered.png')

## 12 · Final Summary

In [ ]:
print('[Summary] ============================================================')
print('[Summary] FULL PIPELINE COMPARISON')
print('[Summary] ============================================================')

for method in ['BM25','BGE','BGE + Reranker','Filter + BGE + Reranker']:
    df   = all_methods[method]
    n10  = df[df['k']==10]['ndcg'].mean()
    p10  = df[df['k']==10]['precision'].mean()
    r1k  = df[df['k']==1000]['recall'].mean()
    print(f'\n[Summary] {method}')
    print(f'  NDCG@10    : {n10:.3f}')
    print(f'  Prec@10    : {p10:.3f}')
    print(f'  Recall@1000: {r1k:.3f}')

print(f'\n[Summary] Latency:')
print(f'  BGE alone              : {BGE_ALONE_MS:.1f}ms')
print(f'  BGE + Reranker         : {BGE_RERANKER_MS:.1f}ms')
print(f'  Filter + BGE + Reranker: {avg_total:.1f}ms')
print('[Summary] ============================================================')